# New FlashGPT Implementation
- Similar to the base model but with more detailed comments
- Slightly different implementation style

In [ ]:
import os
import time
import math
import logging
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
from tqdm.auto import tqdm

# Setting up our computing beast – CUDA or not, we’ll make it work!
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
torch.manual_seed(42)  # for reproducibility (fingers crossed!)
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

logging.basicConfig(
    filename='training_log.txt', 
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger()


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- RMSNorm (instead of LayerNorm, because why not keep it fresh) --- #

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-4):  # using a slightly bigger epsilon for safety
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        norm_x = x.norm(2, dim=-1, keepdim=True)
        rms_x = norm_x * (1.0 / math.sqrt(x.shape[-1]))
        return (x / (rms_x + self.eps)) * self.weight

# --- SwiGLU Feed-Forward Block (because plain old MLPs are so last year) --- #

class SwiGLU(nn.Module):
    """
    SwiGLU feed-forward block.
    Projects input to a doubled dimension, splits, applies SiLU, gates, then projects back.
    """
    def __init__(self, hidden_dim: int, expansion_factor: int = 4, dropout_prob: float = 0.1):
        super().__init__()
        self.expanded_dim = expansion_factor * hidden_dim
        self.fc_in = nn.Linear(hidden_dim, 2 * self.expanded_dim)
        self.fc_out = nn.Linear(self.expanded_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout_prob)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_proj = self.fc_in(x)
        x1, x2 = x_proj.chunk(2, dim=-1)
        x_out = F.silu(x1) * x2  # gating mechanism (pretty neat, right?)
        x_out = self.fc_out(x_out)
        return self.dropout(x_out)

# --- ALiBi Positional Bias (to give that extra oomph) --- #

def build_alibi_tensor(batch_size: int, n_heads: int, seq_len: int, device: torch.device) -> torch.Tensor:
    def get_slopes(n: int):
        def get_slopes_power_of_2(n: int):
            start = 2 ** (-8.0 / n)
            ratio = start
            return [start * (ratio ** i) for i in range(n)]
        if math.log2(n).is_integer():
            return get_slopes_power_of_2(n)
        else:
            closest_power_of_2 = 2 ** math.ceil(math.log2(n))
            slopes = get_slopes_power_of_2(closest_power_of_2)
            return slopes[:n]

    slopes = torch.tensor(get_slopes(n_heads), device=device).unsqueeze(-1).unsqueeze(-1)
    arange_tensor = torch.arange(seq_len, device=device).unsqueeze(0).unsqueeze(0)
    alibi = slopes * arange_tensor
    return alibi

# --- GPTConfig class for configuration sanity --- #

class GPTConfig:
    def __init__(
        self,
        vocab_size: int,
        max_seq_len: int,
        n_embd: int,
        n_layer: int,
        n_head: int,
        dropout_prob: float = 0.1,
        alibi: bool = True,
        flash_attention: bool = True
    ):
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len
        self.n_embd = n_embd
        self.n_layer = n_layer
        self.n_head = n_head
        self.dropout_prob = dropout_prob
        self.alibi = alibi
        self.flash_attention = flash_attention

# --- MultiHeadSelfAttention with ALiBi and optional FlashAttention --- #

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        assert self.n_embd % self.n_head == 0, "Embedding dim must be divisible by number of heads"
        self.head_dim = self.n_embd // self.n_head
        
        self.q_proj = nn.Linear(self.n_embd, self.n_embd)
        self.k_proj = nn.Linear(self.n_embd, self.n_embd)
        self.v_proj = nn.Linear(self.n_embd, self.n_embd)
        self.out_proj = nn.Linear(self.n_embd, self.n_embd)
        self.dropout = nn.Dropout(config.dropout_prob)
        self.flash_attention = config.flash_attention
        
        self.register_buffer("causal_mask", torch.tril(torch.ones(config.max_seq_len, config.max_seq_len)), persistent=False)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None, alibi_bias: torch.Tensor = None) -> torch.Tensor:
        B, T, C = x.size()
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        if self.flash_attention and alibi_bias is None and attention_mask is None:
            attn_output = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout.p, is_causal=True)
        else:
            attn_scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
            causal_mask = self.causal_mask[:T, :T]
            attn_scores = attn_scores.masked_fill(causal_mask == 0, float('-inf'))
            if alibi_bias is not None:
                attn_scores = attn_scores + alibi_bias[:, :, :T]
            if attention_mask is not None:
                extended_mask = attention_mask.unsqueeze(1).unsqueeze(2)
                attn_scores = attn_scores.masked_fill(extended_mask == 0, float('-inf'))
            attn_weights = F.softmax(attn_scores, dim=-1)
            attn_weights = self.dropout(attn_weights)
            attn_output = torch.matmul(attn_weights, v)

        attn_output = attn_output.transpose(1, 2).contiguous().view(B, T, C)
        output = self.out_proj(attn_output)
        return self.dropout(output)

# --- Transformer Block (RMSNorm & SwiGLU) --- #

class TransformerBlock(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.attn_norm = RMSNorm(config.n_embd)
        self.ffn_norm = RMSNorm(config.n_embd)
        self.attn = MultiHeadSelfAttention(config)
        self.mlp = SwiGLU(config.n_embd, expansion_factor=4, dropout_prob=config.dropout_prob)
        self.dropout = nn.Dropout(config.dropout_prob)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None, alibi_bias: torch.Tensor = None) -> torch.Tensor:
        normed_x = self.attn_norm(x)
        attn_out = self.attn(normed_x, attention_mask=attention_mask, alibi_bias=alibi_bias)
        x = x + attn_out
        normed_x2 = self.ffn_norm(x)
        ffn_out = self.mlp(normed_x2)
        return x + ffn_out

# --- The main GPT model --- #

class GPTModel(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.vocab_size = config.vocab_size
        self.max_seq_len = config.max_seq_len
        self.n_embd = config.n_embd
        self.n_layer = config.n_layer
        self.n_head = config.n_head
        
        self.wte = nn.Embedding(self.vocab_size, self.n_embd)
        self.drop = nn.Dropout(config.dropout_prob)
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(self.n_layer)])
        self.norm_f = RMSNorm(config.n_embd)
        
        self.apply(self._init_weights)

    def _init_weights(self, module: nn.Module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0, std=0.02)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None) -> torch.Tensor:
        B, T = input_ids.shape
        assert T <= self.max_seq_len, "Sequence length exceeds model's maximum."
        token_embeddings = self.wte(input_ids)
        x = self.drop(token_embeddings)
        alibi_bias = None
        if self.config.alibi:
            alibi_bias = build_alibi_tensor(B, self.n_head, T, device=x.device)
        
        from torch.utils.checkpoint import checkpoint
        for block in self.blocks:
            if self.training:
                x = checkpoint(block, x, attention_mask, alibi_bias)
            else:
                x = block(x, attention_mask=attention_mask, alibi_bias=alibi_bias)
        return self.norm_f(x)

# --- GPT Head: tying LM head to embeddings --- #

class GPTLMHeadModel(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.transformer = GPTModel(config)
        self.vocab_size = config.vocab_size
        self.n_embd = config.n_embd

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None, labels: torch.Tensor = None) -> dict:
        hidden_states = self.transformer(input_ids, attention_mask=attention_mask)
        logits = F.linear(hidden_states, self.transformer.wte.weight)  # LM head tied to embeddings

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        return {"loss": loss, "logits": logits}


In [ ]:
import os
import requests
import zipfile
from transformers import GPT2Tokenizer
from torch.utils.data import DataLoader, Dataset, random_split
import torch

# Constants
DATA_URL = "https://dldata-public.s3.us-east-2.amazonaws.com/simplebooks.zip"
DATA_DIR = "./simplebooks_data"
MAX_DATA_SIZE = 1 * 1024 * 1024 * 1024  # 1 GB
BATCH_SIZE = 8
MAX_LENGTH = 512  # Maximum token length for GPT-2

# Function to download and extract the dataset
def download_and_extract_data(url, dest_dir):
    if not os.path.exists(dest_dir):
        os.makedirs(dest_dir)
    zip_path = os.path.join(dest_dir, "simplebooks.zip")
    if not os.path.exists(zip_path):
        print("Downloading dataset...")
        response = requests.get(url)
        with open(zip_path, "wb") as f:
            f.write(response.content)
        print("Download complete.")
    else:
        print("Dataset already downloaded.")
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(dest_dir)
    print("Extraction complete.")
    return os.path.join(dest_dir, "simplebooks", "simplebooks-92-raw", "train.txt")

# Custom Dataset class with lazy tokenization to reduce RAM usage
class SimpleBooksDataset(Dataset):
    def __init__(self, file_path, tokenizer, max_length=MAX_LENGTH):
        self.tokenizer = tokenizer
        self.max_length = max_length
        with open(file_path, 'r', encoding='utf-8') as f:
            self.lines = f.readlines()

    def __len__(self):
        return len(self.lines)

    def __getitem__(self, idx):
        line = self.lines[idx]
        tokens = self.tokenizer(line, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        input_ids = tokens['input_ids'].squeeze()
        attention_mask = tokens['attention_mask'].squeeze()
        return {'input_ids': input_ids, 'attention_mask': attention_mask}

# Initialize the GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

# Download and prepare the dataset
data_file = download_and_extract_data(DATA_URL, DATA_DIR)
dataset = SimpleBooksDataset(data_file, tokenizer)

# Split the dataset into training and validation sets (90% train, 10% validation)
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Create DataLoader objects
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

print('Data loaded and tokenized successfully.')


In [ ]:
# Initialize the Custom GPT Model (tweak these parameters as you see fit)      #

config = GPTConfig(
    vocab_size=tokenizer.vocab_size,
    max_seq_len=512,
    n_embd=256,
    n_layer=64,
    n_head=8,
    dropout_prob=0.1,
    alibi=True,
    flash_attention=False
)

model = GPTLMHeadModel(config)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# If running on CPU, compile for speed (DeepSeek-inspired magic!)
if device.type != 'cuda':
    print("Compiling model for CPU acceleration using torch.compile (DeepSeek optimization activated)!")
    try:
        model = torch.compile(model, mode='reduce-overhead')
    except Exception as e:
        print("Compilation failed, continuing without compile. Error:", e)

total_params = sum(p.numel() for p in model.parameters())
print(f'Total Parameters: {total_params}')
print('Model initialized successfully.')


In [ ]:
from torch.optim import AdamW

# Setting up the optimizer – be gentle with the learning rate
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

# LambdaLR scheduler with linear decay
num_epochs = 3
num_training_steps = len(train_dataloader) * num_epochs
num_warmup_steps = 0

def lr_lambda(current_step):
    if current_step < num_warmup_steps:
        return float(current_step) / float(max(1, num_warmup_steps))
    return max(0.0, float(num_training_steps - current_step) / float(max(1, num_training_steps - num_warmup_steps)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


In [ ]:
def load_latest_checkpoint(model, optimizer, scheduler, checkpoint_dir='./checkpoints'):
    if not os.path.exists(checkpoint_dir):
        print(f"No checkpoint directory found at '{checkpoint_dir}'. Starting training from scratch.")
        return model, optimizer, scheduler, 0, 0

    checkpoints = [os.path.join(checkpoint_dir, ckpt) for ckpt in os.listdir(checkpoint_dir) if ckpt.endswith('.pt')]
    if not checkpoints:
        print(f"No checkpoints found in '{checkpoint_dir}'. Starting training from scratch.")
        return model, optimizer, scheduler, 0, 0

    latest_ckpt = max(checkpoints, key=os.path.getctime)
    checkpoint = torch.load(latest_ckpt, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'], strict=False)
    try:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    except ValueError as e:
        print(f"Warning: Optimizer state dict mismatch - {e}. Skipping optimizer state load.")
    try:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    except ValueError as e:
        print(f"Warning: Scheduler state dict mismatch - {e}. Skipping scheduler state load.")
    epoch = checkpoint['epoch']
    global_step = checkpoint['global_step']
    print(f"Loaded checkpoint '{latest_ckpt}' from epoch {epoch+1}, step {global_step}.")
    return model, optimizer, scheduler, epoch + 1, global_step


In [ ]:
# Training Loop( ignoring NaN loss, side result of optimization)   

epochs = 3
checkpoint_interval = 120  # in seconds
start_epoch = 0
global_step = 0
last_checkpoint_time = time.time()
checkpoint_dir = './checkpoints'

print('Starting training...')
model.train()

for epoch in range(start_epoch, epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    epoch_iterator = tqdm(train_dataloader, desc="Training")
    for batch in epoch_iterator:
        inputs = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids=inputs, attention_mask=attention_mask, labels=inputs)
        loss = outputs["loss"]

        # Skip this step if loss is NaN – we just reset and move on
        if torch.isnan(loss):
            print(f"NaN loss encountered at step {global_step}. Skipping update and resetting gradients.")
            logger.warning(f"NaN loss at step {global_step}")
            optimizer.zero_grad()
            continue

        try:
            with torch.autograd.detect_anomaly():
                loss.backward()
        except RuntimeError as e:
            print(f"Runtime error during backward pass at step {global_step}: {e}")
            logger.error(f"Backward error at step {global_step}: {e}")
            optimizer.zero_grad()
            continue

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        global_step += 1
        epoch_iterator.set_postfix(loss=loss.item())

        if time.time() - last_checkpoint_time >= checkpoint_interval:
            ckpt_path = os.path.join(checkpoint_dir, f'checkpoint-epoch{epoch+1}-step{global_step}.pt')
            torch.save({
                'epoch': epoch,
                'global_step': global_step,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
            }, ckpt_path)
            print(f"Saved checkpoint at step {global_step}")
            logger.info(f"Saved checkpoint at step {global_step}")
            last_checkpoint_time = time.time()

    # Run validation after each epoch
    model.eval()
    val_losses = []
    generated_outputs = []
    expected_outputs = []
    with torch.no_grad():
        for batch in val_dataloader:
            inputs = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids=inputs, attention_mask=attention_mask, labels=inputs)
            val_losses.append(outputs["loss"].item())

            sample_input = inputs[0:1]
            generated_ids = sample_input
            for _ in range(50):
                logits = F.linear(model.transformer(generated_ids), model.transformer.wte.weight)
                next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
                generated_ids = torch.cat((generated_ids, next_token), dim=1)
            generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
            expected_text = tokenizer.decode(inputs[0], skip_special_tokens=True)
            generated_outputs.append(generated_text)
            expected_outputs.append(expected_text)
    avg_val_loss = sum(val_losses) / len(val_losses)
    print(f"Validation Loss after epoch {epoch+1}: {avg_val_loss}")
    logger.info(f"Epoch {epoch+1} - Validation Loss: {avg_val_loss}")

    for i, (gen, exp) in enumerate(zip(generated_outputs[:3], expected_outputs[:3])):
        log_str = f"Sample {i+1}:\nExpected: {exp}\nGenerated: {gen}\n{'-'*20}"
        print(log_str)
        logger.info(log_str)

    model.train()

print('Training complete!')


In [ ]:
# Checkpoint Loading and Text Generation (let's have some fun with it!)      #
import torch
import torch.nn.functional as F

def generate_text(prompt, model, tokenizer, max_length=100, temperature=1.0, top_k=50):
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    generated = input_ids
    with torch.no_grad():
        for _ in range(max_length - input_ids.size(1)):
            outputs = model(generated)
            logits = outputs['logits'][:, -1, :] / temperature
            filtered_logits = top_k_filtering(logits, top_k=top_k)
            probabilities = F.softmax(filtered_logits, dim=-1)
            next_token = torch.multinomial(probabilities, num_samples=1)
            generated = torch.cat((generated, next_token), dim=1)
            if next_token.item() == tokenizer.eos_token_id:
                break
    generated_text = tokenizer.decode(generated[0], skip_special_tokens=True)
    return generated_text

def top_k_filtering(logits, top_k=50):
    if top_k > 0:
        values, _ = torch.topk(logits, top_k)
        min_values = values[:, -1].unsqueeze(1)
        logits = torch.where(logits < min_values, torch.full_like(logits, float('-inf')), logits)
    return logits


def load_latest_checkpoint_simple(model, optimizer, scheduler, checkpoint_dir='./checkpoints'):
    checkpoints = [os.path.join(checkpoint_dir, ckpt) for ckpt in os.listdir(checkpoint_dir) if ckpt.endswith('.pt')]
    if not checkpoints:
        print('No checkpoints found.')
        return 0
    latest_ckpt = max(checkpoints, key=os.path.getctime)
    checkpoint = torch.load(latest_ckpt, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'], strict=False)
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    print(f"Loaded checkpoint from {latest_ckpt} at step {checkpoint['global_step']}")
    return checkpoint['global_step']

global_step = load_latest_checkpoint_simple(model, optimizer, scheduler)
prompt = "Once upon a time, there was a boy named Jack. Jack was very curious about the world and always wanted to explore new places. One day, he found a mysterious key" 
print("Prompt:", prompt)
print("Generated Text:", generate_text(prompt, model, tokenizer))


In [ ]:
import fitz  # PyMuPDF

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

pdf_path = './sample.pdf'
static_dir = './static'
os.makedirs(static_dir, exist_ok=True)

pdf_text = extract_text_from_pdf(pdf_path)

inputs = tokenizer(pdf_text, return_tensors='pt', max_length=512, truncation=True, padding='max_length')

print('Starting training on PDF text...')
model.train()

last_checkpoint_time = time.time()
epochs_pdf = 5

for epoch in range(epochs_pdf):
    print(f"PDF Training Epoch {epoch+1}/{epochs_pdf}")
    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)

    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
    loss = outputs["loss"]
    
    if torch.isnan(loss):
        print(f"NaN loss encountered during PDF training at epoch {epoch+1}. Skipping update.")
        optimizer.zero_grad()
        continue

    try:
        with torch.autograd.detect_anomaly():
            loss.backward()
    except RuntimeError as e:
        print(f"Backward error during PDF training at epoch {epoch+1}: {e}")
        optimizer.zero_grad()
        continue

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad()

    global_step += 1
    print(f"PDF Epoch {epoch+1}, Loss: {loss.item()}")

    if time.time() - last_checkpoint_time >= checkpoint_interval:
        ckpt_path = os.path.join(checkpoint_dir, f'pdf_checkpoint-epoch{epoch+1}-step{global_step}.pt')
        torch.save({
            'epoch': epoch,
            'global_step': global_step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
        }, ckpt_path)
        print(f"Saved PDF checkpoint at step {global_step}")
        last_checkpoint_time = time.time()

    model.eval()
    val_losses = []
    with torch.no_grad():
        for batch in val_dataloader:
            input_ids_val = batch['input_ids'].to(device)
            attention_mask_val = batch['attention_mask'].to(device)
            outputs_val = model(input_ids=input_ids_val, attention_mask=attention_mask_val, labels=input_ids_val)
            val_losses.append(outputs_val["loss"].item())
    avg_val_loss = sum(val_losses) / len(val_losses)
    print(f"Validation Loss after PDF Epoch {epoch+1}: {avg_val_loss}")
    model.train()

print('PDF Training complete!')
